# Retrain THIS repo's MuJoCo envs (modern stack) — PointMaze_Large + AntMaze

Retrain the two under-trained MuJoCo checkpoints of this reimplementation so our
MCTS / execution-feedback experiments can run on them (like Fetch, already READY).

**Which notebook is this?** THIS = *this repo, modern stack* (`mujoco>=3` +
`gymnasium-robotics`), substitute envs → checkpoints usable by `scripts/run_e1*.py`.
For the **paper's EXACT envs** (advisor requirement, old `mujoco_py 2.0` stack) use
`repro/kaggle_notebook.md` instead.

Modern stack installs cleanly via pip — no old-CUDA / mujoco-py-2.0 pain. GPU is
optional (nets are tiny; MuJoCo CPU stepping + the planner dominate) → a **CPU session**
(more weekly hours) is often the better choice here.

## Cell 1 — get this repo (use YOUR fork URL)

In [ ]:
REPO_URL = "https://github.com/<YOUR_FORK>/latent-lanmarks"  # <-- edit
import os
if not os.path.isdir('/kaggle/working/latent-lanmarks'):
    !git clone -q $REPO_URL /kaggle/working/latent-lanmarks
%cd /kaggle/working/latent-lanmarks

## Cell 2 — deps (modern, one-shot) + sanity

In [ ]:
!pip -q install "torch>=2.0" "numpy>=1.23" "mujoco>=3" gymnasium gymnasium-robotics matplotlib
import mujoco, gymnasium as gym, gymnasium_robotics, torch
gym.register_envs(gymnasium_robotics)
print('mujoco', mujoco.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
e = gym.make('PointMaze_Large-v3'); print('env OK', e.observation_space); e.close()

## Cell 3 — restore prior checkpoints from a Dataset (skip on first session)

In [ ]:
import os, shutil, glob
os.makedirs('checkpoint', exist_ok=True)
# attach your checkpoints Dataset under /kaggle/input/<name>/ then adjust this glob:
for src in glob.glob('/kaggle/input/*/l3p_pointmaze_mujoco.pt') + glob.glob('/kaggle/input/*/l3p_AntMaze.pt'):
    shutil.copy(src, 'checkpoint/' + os.path.basename(src)); print('restored', src)
print('have:', os.listdir('checkpoint'))

## Cell 4 — retrain PointMaze (MuJoCo) — the tractable one
Re-run this cell each session; it resumes if a checkpoint exists (step counters preserved).

In [ ]:
import os
ck = 'checkpoint/l3p_pointmaze_mujoco.pt'
resume = f'--resume {ck}' if os.path.exists(ck) else ''
!python scripts/train.py --env PointMazeMuJoCo --steps 3000000 --workers 3 \
    --save-every 200000 --save {ck} {resume} \
    --log-file logs/pointmaze_mujoco_retrain.log

## Cell 5 — retrain AntMaze — the hard one (locomotion)
WATCH the eval-success line. AntMaze must first LEARN TO WALK from sparse reward
(chicken-and-egg). If eval stays 0.00 after ~1-2M steps, STOP — more of the same budget
won't break the deadlock; needs dense-reward shaping or a smaller maze. To try the
easier maze, edit `l3p/envs/mujoco.py` `_GYMNASIUM_IDS`: `"AntMaze" -> "AntMaze_UMaze-v5"`.

In [ ]:
import os
ck = 'checkpoint/l3p_AntMaze.pt'
resume = f'--resume {ck}' if os.path.exists(ck) else ''
!python scripts/train.py --env AntMaze --steps 3000000 --workers 3 \
    --save-every 200000 --save {ck} {resume} \
    --log-file logs/antmaze_retrain.log

## Cell 6 — readiness check + persist

In [ ]:
!python scripts/check_checkpoint_readiness.py --env PointMazeMuJoCo --load checkpoint/l3p_pointmaze_mujoco.pt --episodes 20
!python scripts/check_checkpoint_readiness.py --env AntMaze --load checkpoint/l3p_AntMaze.pt --episodes 20
# Commit the notebook -> /kaggle/working is saved as output. Publish checkpoint/*.pt as a
# Dataset and point Cell 3 at it next session to resume.

## Cell 7 (optional) — once READY, run the MCTS ablation here

In [ ]:
!python scripts/run_e1c.py --env PointMazeMuJoCo --load checkpoint/l3p_pointmaze_mujoco.pt \
    --seeds 0 1 --episodes 30 --sigmas 0 0.1 0.3 --out logs/pmmj_e1c.json --plot logs/pmmj_e1c.png

## Operating notes
- **Cross-session:** `--save-every 200000` writes periodically; `--resume <ckpt>` continues
  (step counters preserved unless `--reset-counters`). Persist `checkpoint/*.pt` to a Kaggle
  Dataset between sessions.
- **CPU vs GPU:** env stepping dominates → a CPU session (more weekly hours) usually beats
  burning the ~30h/week GPU quota here.
- **`--workers` here** = in-process VecEnv (sequential, not MPI): more workers add data
  diversity but do NOT speed wall-clock. Keep at ~3.
- **Honest expectation:** PointMaze_Large likely becomes usable (planner ≳ 0.3); AntMaze may
  stay ~0 (locomotion deadlock) even with more budget on the *substitute* env — the paper
  cracked it with 12 MPI workers + 3M steps + their custom env. Do PointMaze first.